# Fine-tune Cross-Encoder v0.6 - Ensemble 5 Runs with Different Seeds

Ensemble approach: Train 5 models với seed khác nhau, aggregate predictions.

| | |
|---|---|
| **Dataset** | v0.5 — 9,350 train / 2,000 validation / 2,000 test (13,350 pairs, balanced 20% per class) |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `MSELoss` |
| **Evaluator** | Spearman correlation |
| **Epochs** | 15 (locked from Phase 2.1) |
| **Batch size** | 16 (locked from Phase 2.1) |
| **Learning rate** | 5e-5 (optimal from Phase 2.1) |
| **Seeds** | [42, 123, 456, 789, 999] |
| **Branch** | `experiment/cross-encoder-v0.6` |

**Goal**: Test if ensemble of 5 models improves test LabelAcc over single model (65.15%).

**Expected**: Reduce variance, +0.1-0.5pp improvement via averaging → 65.2-65.7%.

In [ ]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.6
!git pull origin experiment/cross-encoder-v0.6

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Helper Functions

In [ ]:
import json
import torch
import torch.nn.functional as F
import numpy as np
from scipy.stats import spearmanr
from sentence_transformers import CrossEncoder, InputExample
from typing import Any
import random

# Evaluator: CECorrelationEvaluator (Spearman)
class CECorrelationEvaluator:
    """Evaluate cross-encoder using Spearman correlation."""
    def __init__(self, sentence_pairs: list, labels_0_1: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_1 = labels_0_1
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = "") -> "CECorrelationEvaluator":
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_1=[ex.label for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        corr, _ = spearmanr(preds, self.labels_0_1)
        return float(corr) if not np.isnan(corr) else 0.0

def load_jsonl(path: str) -> list[dict[str, Any]]:
    """Load JSONL file."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def load_dataset(data_dir: str) -> tuple[list[InputExample], list[InputExample], list[InputExample]]:
    """Load train/val/test InputExample lists."""
    train_data = load_jsonl(f"{data_dir}/cross_encoder_train.jsonl")
    val_data = load_jsonl(f"{data_dir}/cross_encoder_validation.jsonl")
    test_data = load_jsonl(f"{data_dir}/cross_encoder_test.jsonl")

    def to_input_examples(records):
        return [
            InputExample(texts=[rec['cv_text'], rec['jd_text']], label=rec['label'])
            for rec in records
        ]

    return to_input_examples(train_data), to_input_examples(val_data), to_input_examples(test_data)

def compute_metrics(model: CrossEncoder, examples: list[InputExample], batch_size: int = 32) -> dict:
    """Compute MAE, RMSE, LabelAcc on examples."""
    preds = model.predict([ex.texts for ex in examples], batch_size=batch_size, show_progress_bar=False)
    preds_100 = np.asarray(preds) * 100
    labels_100 = np.array([ex.label * 100 for ex in examples])

    mae = np.mean(np.abs(preds_100 - labels_100))
    rmse = np.sqrt(np.mean((preds_100 - labels_100) ** 2))
    label_acc = np.mean(np.abs(preds_100 - labels_100) <= 10)

    return {'MAE': mae, 'RMSE': rmse, 'LabelAcc': label_acc}

print("✅ Helper functions loaded.")

## Load Dataset (Once)

In [ ]:
train_examples, val_examples, test_examples = load_dataset('datasets/versions/v0.5/cross_encoder')

print(f"Train: {len(train_examples)} pairs")
print(f"Val:   {len(val_examples)} pairs")
print(f"Test:  {len(test_examples)} pairs")
print(f"Total: {len(train_examples) + len(val_examples) + len(test_examples)} pairs")

## Ensemble: 5 Runs with Different Seeds

In [ ]:
import torch
from torch.utils.data import DataLoader
import os
import shutil

# Configuration
base_model = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
learning_rate = 5e-5
epochs = 15
batch_size = 16
seeds = [42, 123, 456, 789, 999]

# Calculate warmup steps (fixed for all runs)
total_steps = (len(train_examples) // batch_size + 1) * epochs
warmup_steps = int(total_steps * 0.1)

print(f"📊 Ensemble Configuration:")
print(f"  Base model: {base_model}")
print(f"  Learning rate: {learning_rate:.0e}")
print(f"  Epochs: {epochs}")
print(f"  Batch size: {batch_size}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Seeds: {seeds}")
print(f"  Total steps per run: {total_steps}")
print()

# Store results from all 5 runs
all_results = []
all_predictions = []  # Store predictions from each model

for run_idx, seed in enumerate(seeds, 1):
    print(f"\n{'='*70}")
    print(f"🚀 Run {run_idx}/5: seed={seed}")
    print(f"{'='*70}")
    
    # Set seed for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    run_name = f"v0.6-ensemble-run{run_idx}-seed{seed}"
    output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"
    
    # Initialize model
    model = CrossEncoder(
        base_model,
        num_labels=1,
        default_activation_function=torch.nn.Sigmoid()
    )
    
    # Setup evaluator and data
    evaluator = CECorrelationEvaluator.from_input_examples(val_examples, name="val_spearman")
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)
    
    # Train
    model.fit(
        train_dataloader=train_dataloader,
        evaluator=evaluator,
        epochs=epochs,
        loss_fct=torch.nn.MSELoss(),
        optimizer_params={'lr': learning_rate},
        warmup_steps=warmup_steps,
        output_path=output_dir,
        save_best_model=True,
        use_amp=True,
        max_grad_norm=1.0,
        show_progress_bar=True
    )
    
    # Load best model and evaluate
    best_model = CrossEncoder(output_dir)
    val_metrics = compute_metrics(best_model, val_examples)
    test_metrics = compute_metrics(best_model, test_examples)
    
    print(f"\n✅ Run {run_idx} completed (seed={seed})")
    print(f"  Val LabelAcc:  {val_metrics['LabelAcc']:.4f} ({val_metrics['LabelAcc']*100:.2f}%)")
    print(f"  Test LabelAcc: {test_metrics['LabelAcc']:.4f} ({test_metrics['LabelAcc']*100:.2f}%)")
    
    # Store results
    result = {
        'run': run_idx,
        'seed': seed,
        'run_name': run_name,
        'val_labelacc': float(val_metrics['LabelAcc']),
        'test_labelacc': float(test_metrics['LabelAcc']),
        'val_mae': float(val_metrics['MAE']),
        'test_mae': float(test_metrics['MAE']),
        'val_rmse': float(val_metrics['RMSE']),
        'test_rmse': float(test_metrics['RMSE'])
    }
    all_results.append(result)
    
    # Get predictions for ensemble
    test_preds = best_model.predict([ex.texts for ex in test_examples], batch_size=32, show_progress_bar=False)
    all_predictions.append(test_preds)

print(f"\n{'='*70}")
print(f"✅ All 5 runs completed!")
print(f"{'='*70}")

## Ensemble Aggregation

In [ ]:
# Average predictions from 5 models
ensemble_preds = np.mean(all_predictions, axis=0)
ensemble_preds_100 = ensemble_preds * 100
test_labels_100 = np.array([ex.label * 100 for ex in test_examples])

# Compute ensemble metrics
ensemble_mae = np.mean(np.abs(ensemble_preds_100 - test_labels_100))
ensemble_rmse = np.sqrt(np.mean((ensemble_preds_100 - test_labels_100) ** 2))
ensemble_label_acc = np.mean(np.abs(ensemble_preds_100 - test_labels_100) <= 10)

print(f"\n📊 Individual Runs Summary:")
print(f"{'Run':<6} {'Seed':<8} {'Val Acc':<12} {'Test Acc':<12}")
print(f"{'-'*40}")
for result in all_results:
    print(f"{result['run']:<6} {result['seed']:<8} {result['val_labelacc']*100:>10.2f}% {result['test_labelacc']*100:>10.2f}%")

print(f"\n📊 Ensemble Results:")
print(f"  Ensemble Test LabelAcc: {ensemble_label_acc:.4f} ({ensemble_label_acc*100:.2f}%)")
print(f"  Ensemble Test MAE:      {ensemble_mae:.4f}")
print(f"  Ensemble Test RMSE:     {ensemble_rmse:.4f}")

print(f"\n📈 Comparison:")
baseline = 0.6515
print(f"  Baseline (Phase 2.1, single run): {baseline*100:.2f}%")
print(f"  Ensemble Average:                  {ensemble_label_acc*100:.2f}%")
improvement = (ensemble_label_acc - baseline) * 100
print(f"  Improvement: {improvement:+.2f}pp")

print(f"\n📊 Statistics:")
test_accs = [r['test_labelacc'] for r in all_results]
print(f"  Best single run:  {np.max(test_accs)*100:.2f}%")
print(f"  Worst single run: {np.min(test_accs)*100:.2f}%")
print(f"  Std dev:          {np.std(test_accs)*100:.2f}pp")
print(f"  Mean:             {np.mean(test_accs)*100:.2f}%")

## Save Reports

In [ ]:
import os
import json

os.makedirs('artifacts/reports', exist_ok=True)

# Save individual run reports
for result in all_results:
    report = {
        "experiment": "v0.6 Ensemble - Individual Run",
        "base_model": base_model,
        "dataset_version": "v0.5",
        "run": result['run_name'],
        "seed": result['seed'],
        "loss": "MSE",
        "evaluator": "Spearman",
        "epochs": epochs,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "metrics": {
            "validation": {
                "LabelAcc": result['val_labelacc'],
                "MAE": result['val_mae'],
                "RMSE": result['val_rmse']
            },
            "test": {
                "LabelAcc": result['test_labelacc'],
                "MAE": result['test_mae'],
                "RMSE": result['test_rmse']
            }
        },
        "model_path": f"artifacts/models/{output_dir.split('/')[-1]}"
    }
    
    report_path = f'artifacts/reports/fine_tune_cross_encoder_v0.6_ensemble_run{result["run"]}_seed{result["seed"]}_report.json'
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    print(f"✅ Report saved: {report_path}")

# Save ensemble summary report
ensemble_report = {
    "experiment": "Phase 5: Ensemble 5 Runs with Different Seeds",
    "base_model": base_model,
    "dataset_version": "v0.5",
    "loss": "MSE",
    "evaluator": "Spearman",
    "epochs": epochs,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "seeds": seeds,
    "individual_runs": all_results,
    "ensemble_metrics": {
        "test_labelacc": float(ensemble_label_acc),
        "test_mae": float(ensemble_mae),
        "test_rmse": float(ensemble_rmse)
    },
    "statistics": {
        "best_single_run_test_labelacc": float(np.max(test_accs)),
        "worst_single_run_test_labelacc": float(np.min(test_accs)),
        "mean_test_labelacc": float(np.mean(test_accs)),
        "std_dev_test_labelacc": float(np.std(test_accs))
    },
    "comparison_to_baseline": {
        "baseline_test_labelacc": 0.6515,
        "ensemble_improvement_pp": float(improvement)
    }
}

summary_path = 'artifacts/reports/fine_tune_cross_encoder_v0.6_ensemble_5runs_report.json'
with open(summary_path, 'w') as f:
    json.dump(ensemble_report, f, indent=2)

print(f"\n✅ Ensemble report saved: {summary_path}")

## Save to Google Drive (Optional)

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"
os.makedirs(f"{drive_base}/models", exist_ok=True)
os.makedirs(f"{drive_base}/reports", exist_ok=True)

# Save all 5 models
for result in all_results:
    src_dir = f"artifacts/models/cross-encoder-cv-jd-{result['run_name']}"
    dest_dir = f"{drive_base}/models/{result['run_name']}"
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    if os.path.exists(src_dir):
        shutil.copytree(src_dir, dest_dir)
        print(f"✅ Saved model: {result['run_name']}")

# Save all reports
for file in os.listdir('artifacts/reports'):
    if 'ensemble' in file:
        src = f'artifacts/reports/{file}'
        dest = f"{drive_base}/reports/{file}"
        shutil.copy(src, dest)
        print(f"✅ Saved report: {file}")

print(f"\n✅ All models and reports saved to Google Drive!")